In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from tqdm import tqdm

In [3]:
def load_images_and_labels(dataset_path):
    X, y = [], []
    classes = os.listdir(dataset_path)
    for label in classes:
        class_path = os.path.join(dataset_path, label)
        for fname in os.listdir(class_path):
            if fname.lower().endswith(('.jpg', '.png', '.bmp')):
                img_path = os.path.join(class_path, fname)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (128, 128))
                X.append(img)
                y.append(label)
    return X, y

In [4]:
def extract_hog_features(images):
    features = []
    for img in tqdm(images):
        feat = hog(img, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys')
        features.append(feat)
    return np.array(features)

In [5]:
dataset_path = "/content/drive/MyDrive/Flying object detection system/1/UAV_Dataset"
X_raw, y = load_images_and_labels(dataset_path)
X = extract_hog_features(X_raw)

100%|██████████| 4065/4065 [00:27<00:00, 148.55it/s]


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
models = {
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred))

=== SVM ===
              precision    recall  f1-score   support

   Aeroplane       0.93      0.86      0.89       212
        Bird       0.71      0.88      0.79       189
       Drone       0.79      0.71      0.75       214
  Helicopter       0.78      0.75      0.76       198

    accuracy                           0.80       813
   macro avg       0.80      0.80      0.80       813
weighted avg       0.80      0.80      0.80       813

=== Random Forest ===
              precision    recall  f1-score   support

   Aeroplane       0.92      0.78      0.84       212
        Bird       0.60      0.70      0.65       189
       Drone       0.71      0.63      0.67       214
  Helicopter       0.67      0.75      0.71       198

    accuracy                           0.72       813
   macro avg       0.72      0.72      0.72       813
weighted avg       0.73      0.72      0.72       813

=== KNN ===
              precision    recall  f1-score   support

   Aeroplane       0.84      